# 02 — Reading processed ADCP data (CODAS database)

**Data:** Healy cruise `hly2018`, OS150 broadband — processed output  
**Location:** `~/adcpcode/programs/pycurrents_test_data/uhdas_data/proc/os150bb/`  

Run from the `pycodas` environment:
```bash
conda activate pycodas
jupyter notebook
```

---

### Processed vs. raw

| | Raw (notebook 01) | Processed (this notebook) |
|---|---|---|
| **Time resolution** | single ping (~1 s) | 5-min ensemble average |
| **Coordinates** | instrument XYZ | Earth frame (u = east, v = north) |
| **Navigation** | not applied | heading + ship velocity removed |
| **Editing** | none | flagged bad data masked |
| **Read with** | `Multiread` | `get_profiles` |

---
## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from pycurrents.codas import get_profiles

---
## 2. Load the CODAS database

`get_profiles` reads the CODAS binary database from `proc/<instrument>/adcpdb/`.  
The database name (`a_hly`) is the prefix shared by all `.blk` files in that directory.

In [ ]:
proc_dir = Path.home() / 'adcpcode/programs/pycurrents_test_data/uhdas_data/proc/os150bb'
dbname   = proc_dir / 'adcpdb/a_hly'

data = get_profiles(str(dbname))

print(f'Profiles (ensembles): {data.nprofs}')
print(f'Depth bins:           {data.nbins}')
print(f'Depth range:          {data.dep[0]:.1f} m – {data.dep[-1]:.1f} m')
print(f'Time range:  dday     {data.dday.min():.4f} – {data.dday.max():.4f}')

---
## 3. Inspect what's inside

`data.names` lists every variable stored in the object.

In [ ]:
print('Variables:', data.names)

# shape check on key variables
for k in ['u', 'v', 'w', 'umeas', 'vmeas', 'amp1', 'dday', 'dep', 'lon', 'lat']:
    if k in data.names or hasattr(data, k):
        val = getattr(data, k)
        if hasattr(val, 'shape'):
            print(f'  {k:10s}  shape={val.shape}')
        else:
            print(f'  {k:10s}  scalar={val}')

---
## 4. Plot ocean velocities — u (east) and v (north)

`data.u` and `data.v` are in **Earth frame** (heading + ship motion already removed).  
Compare to `data.umeas` / `data.vmeas` which are still relative to the ship.

In [ ]:
dday = data.dday
dep  = data.dep

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, sharey=True)

components = [
    (data.u, 'u  (east)',  'RdBu_r', 1.5),
    (data.v, 'v  (north)', 'RdBu_r', 1.5),
]

for ax, (vel, label, cmap, vlim) in zip(axes, components):
    pcm = ax.pcolormesh(
        dday, dep, vel.T,
        cmap=cmap, vmin=-vlim, vmax=vlim,
        shading='auto'
    )
    ax.set_ylabel('Depth (m)')
    ax.set_title(f'Ocean velocity {label}  [m/s]  — Earth frame')
    plt.colorbar(pcm, ax=ax, label='m/s', pad=0.01)

axes[0].set_ylim(dep[-1], dep[0])
axes[-1].set_xlabel('Decimal day')
fig.suptitle('Healy 2018 — OS150bb — processed CODAS data', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Backscatter — echo amplitude

Processed data stores one amplitude variable per beam: `amp1` through `amp4`.  
Average them for a single backscatter section.

In [ ]:
amp_mean = np.ma.mean(
    np.ma.stack([data.amp1, data.amp2, data.amp3, data.amp4], axis=2),
    axis=2
)

fig, ax = plt.subplots(figsize=(14, 4))

pcm = ax.pcolormesh(
    dday, dep, amp_mean.T,
    cmap='viridis',
    shading='auto'
)
ax.set_ylim(dep[-1], dep[0])
ax.set_xlabel('Decimal day')
ax.set_ylabel('Depth (m)')
ax.set_title('Echo amplitude (beam average) — acoustic backscatter proxy')
plt.colorbar(pcm, ax=ax, label='counts')
plt.tight_layout()
plt.show()

---
## 6. Ship track on a map

`data.lon` and `data.lat` are already interpolated to each ensemble time —  
no separate GPS file needed (unlike with raw data).

In [ ]:
from pycurrents.plot.maptools import mapper

print(f'lon: {np.nanmin(data.lon):.2f} – {np.nanmax(data.lon):.2f}')
print(f'lat: {np.nanmin(data.lat):.2f} – {np.nanmax(data.lat):.2f}')

m = mapper([-163, -153], [18, 25])
m.topo()
m.grid()

valid = ~np.isnan(data.lon) & ~np.isnan(data.lat)
x, y = m(data.lon[valid], data.lat[valid])
m.ax.scatter(x, y, s=4, c='red', zorder=5, label='ship track')
m.ax.legend()
plt.show()

---
## 7. Time subset

Pass `ddrange` to load only part of the cruise — useful for large datasets.

In [ ]:
data_sub = get_profiles(str(dbname), ddrange=(149.5, 149.7))

print(f'Full dataset:   {data.nprofs} profiles')
print(f'Subset (0.2 d): {data_sub.nprofs} profiles')

---
## Variable reference

| variable | shape | description |
|---|---|---|
| `u`, `v` | (nprofs, nbins) | ocean velocity, Earth frame (east/north) |
| `umeas`, `vmeas` | (nprofs, nbins) | velocity relative to ship (before nav removal) |
| `w` | (nprofs, nbins) | vertical velocity |
| `e` | (nprofs, nbins) | error velocity |
| `amp1`–`amp4` | (nprofs, nbins) | echo amplitude per beam |
| `dep` | (nbins,) | depth bin centers (m) |
| `dday` | (nprofs,) | decimal day (zero-based from Jan 1) |
| `lon`, `lat` | (nprofs,) | ship position at end of ensemble |
| `heading` | (nprofs,) | ship heading (°) |

> **Decimal day is zero-based**: Jan 1 at noon = 0.5, not 1.5.  
> **Position is at the END of the ensemble**, not the start.

---
## What's next?

- **03_shear.ipynb** — compute vertical shear `du/dz`, `dv/dz` from processed velocities
- **04_backscatter.ipynb** — calibrate amplitude to volume backscatter strength (Sv)
- **Compare u/v vs umeas/vmeas** — see how much the ship motion contributes